# Test Qwen 27B FP8 — Kernel local

Exécuter les cellules dans l'ordre.

## 1. Environnement

In [ ]:
import os, sys, torch, transformers
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Désactiver DeepGEMM

In [ ]:
os.environ["TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR"] = "1"
print("TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR =", os.environ.get("TRANSFORMERS_DISABLE_DEEPGEMM_LINEAR"))

## 3. Chemin local du modèle

Modifier `MODEL_PATH` avant de continuer.

In [ ]:
MODEL_PATH = "/TON/CHEMIN/LOCAL/QWEN_27B_FP8"
print("MODEL_PATH:", MODEL_PATH)
print("Existe:", os.path.exists(MODEL_PATH))
if os.path.exists(MODEL_PATH):
    for x in os.listdir(MODEL_PATH)[:30]:
        print(" -", x)

## 4. Module FP8

In [ ]:
import transformers.integrations.finegrained_fp8 as fp8
print("Module FP8:", fp8)
print("load_finegrained_fp8_kernel:", fp8.load_finegrained_fp8_kernel)
print("lazy_load_kernel:", fp8.lazy_load_kernel)

## 5. Vérifier le kernel local

`local_fp8_kernel` doit déjà avoir été construit.

In [ ]:
try:
    print("Kernel local:", local_fp8_kernel)
    print("Type:", type(local_fp8_kernel))
    print("matmul:", local_fp8_kernel.matmul)
    print("matmul_batched:", local_fp8_kernel.matmul_batched)
    print("grouped_matmul:", local_fp8_kernel.grouped_matmul)
except NameError:
    print("ERREUR: local_fp8_kernel n'est pas défini.")
    raise

## 6. Patch Transformers FP8

In [ ]:
assert local_fp8_kernel is not None

fp8.load_finegrained_fp8_kernel = lambda: local_fp8_kernel
fp8.lazy_load_kernel = lambda *args, **kwargs: local_fp8_kernel

for name in ["_load_finegrained_fp8_kernel", "load_finegrained_fp8_kernel"]:
    obj = getattr(fp8, name, None)
    if obj is not None and hasattr(obj, "cache_clear"):
        try:
            obj.cache_clear()
            print("Cache vidé:", name)
        except Exception as e:
            print("Cache non vidé:", name, e)

print("Patch Transformers FP8 installé")
print("Kernel local:", local_fp8_kernel)

## 7. Test du loader FP8

In [ ]:
print("=== TEST LOADER FP8 ===")
k = fp8.load_finegrained_fp8_kernel()
print("OK - kernel retourné")
print("Type:", type(k))
print(k)
print("matmul:", k.matmul)
print("matmul_batched:", k.matmul_batched)
print("grouped_matmul:", k.grouped_matmul)

## 8. Processor

In [ ]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=True
)
print("Processor OK:", type(processor))

## 9. Charger Qwen 27B FP8

In [ ]:
from transformers import AutoModelForMultimodalLM
print("Chargement modèle...")
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    trust_remote_code=True,
    device_map="auto",
    dtype="auto"
)
model.eval()
print("MODELE CHARGE:", type(model))

## 10. GPU

In [ ]:
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM allouée: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")
    print(f"VRAM réservée: {torch.cuda.memory_reserved(0)/1024**3:.2f} GB")
try:
    print("Device model:", model.device)
except Exception:
    print("Modèle distribué via device_map")

## 11. Prompt minimal

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type": "text", "text": "Réponds uniquement par le mot BONJOUR."}]
}]
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
print(text)

## 12. Tokenisation

In [ ]:
inputs = processor(text=[text], return_tensors="pt")
print(inputs.keys())
inputs = {
    k: v.to(model.device) if hasattr(v, "to") else v
    for k, v in inputs.items()
}
print("Tokens entrée:", inputs["input_ids"].shape[-1])

## 13. Génération

In [ ]:
print("================================")
print("DEBUT GENERATION QWEN FP8")
print("================================")
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )
print("GENERATION TERMINEE")

## 14. Décodage

In [ ]:
generated = outputs[:, inputs["input_ids"].shape[-1]:]
response = processor.batch_decode(
    generated,
    skip_special_tokens=True
)[0]
print("================================")
print("REPONSE QWEN 27B FP8")
print("================================")
print(response)
print("================================")